In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import shap

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load datasets
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
transl = pd.read_csv('../data/raw/product_category_name_translation.csv')

# Format dates
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Merge to get customer unique ID
orders = orders.merge(customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id')

In [ ]:
# Determine the cutoff date to prevent data leakage.
# We predict if a customer will churn in the LAST 90 DAYS of the dataset.
max_date = orders['order_purchase_timestamp'].max()
cutoff_date = max_date - pd.Timedelta(days=90)

# Split data into "history" (before cutoff) and "future" (after cutoff)
history_orders = orders[orders['order_purchase_timestamp'] < cutoff_date]
future_orders = orders[orders['order_purchase_timestamp'] >= cutoff_date]

# Customers to score: anyone who made a purchase in the history period
valid_customers = history_orders['customer_unique_id'].unique()

# Label: 1 if churned (no purchase in future), 0 if retained (purchased in future)
purchased_in_future = future_orders['customer_unique_id'].unique()

# Create base dataframe
df_model = pd.DataFrame({'customer_unique_id': valid_customers})
df_model['churn'] = (~df_model['customer_unique_id'].isin(purchased_in_future)).astype(int)

print(f"Total customers: {len(df_model)}")
print(f"Churn rate: {df_model['churn'].mean()*100:.2f}%")

In [ ]:
# 1. Recency, Frequency
rfm = history_orders.groupby('customer_unique_id').agg(
    last_purchase=('order_purchase_timestamp', 'max'),
    frequency=('order_id', 'nunique')
).reset_index()
rfm['recency'] = (cutoff_date - rfm['last_purchase']).dt.days
rfm = rfm.drop(columns=['last_purchase'])

# 2. Monetary
monetary = history_orders.merge(payments, on='order_id').groupby('customer_unique_id').agg(
    monetary=('payment_value', 'sum')
).reset_index()

# 3. Average Review Score
rev = history_orders.merge(reviews, on='order_id').groupby('customer_unique_id').agg(
    avg_review_score=('review_score', 'mean')
).reset_index()

# 4. State
state = history_orders.groupby('customer_unique_id').agg(
    state=('customer_state', 'first')
).reset_index()

# 5. Dominant Category
cat_data = history_orders.merge(items, on='order_id').merge(products, on='product_id').merge(transl, on='product_category_name')
dominant_cat = cat_data.groupby('customer_unique_id').agg(
    category=('product_category_name_english', lambda x: x.mode()[0] if not x.empty else 'unknown')
).reset_index()

# Merge all features
for feature_df in [rfm, monetary, rev, state, dominant_cat]:
    df_model = df_model.merge(feature_df, on='customer_unique_id', how='left')

# Fill missing reviews and category
df_model['avg_review_score'] = df_model['avg_review_score'].fillna(df_model['avg_review_score'].median())
df_model['category'] = df_model['category'].fillna('unknown')
df_model['monetary'] = df_model['monetary'].fillna(0)

df_model.head()

In [ ]:
# Features and Target
X = df_model.drop(columns=['customer_unique_id', 'churn'])
y = df_model['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

In [ ]:
# Preprocessing pipelines
numeric_features = ['recency', 'frequency', 'monetary', 'avg_review_score']
categorical_features = ['category', 'state']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# For categorical features, we only keep top categories to avoid huge dimensionality
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', max_categories=20))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Define the models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(), random_state=42, n_jobs=-1)
}

In [ ]:
# Train and evaluate models
results = {}
best_auc = 0
best_model_name = None
best_pipeline = None

plt.figure(figsize=(10, 8))

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', model)])
    
    pipeline.fit(X_train, y_train)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_pred_proba)
    results[name] = auc
    
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
    
    if auc > best_auc:
        best_auc = auc
        best_model_name = name
        best_pipeline = pipeline

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-AUC Curve for Churn Models')
plt.legend()
plt.show()

print(f"\nBest Model: {best_model_name} with AUC = {best_auc:.3f}")

In [ ]:
# SHAP Interpretation for the best model
print(f"Explaining {best_model_name} with SHAP...")

# Extract preprocessor and model from pipeline
prep = best_pipeline.named_steps['preprocessor']
clf = best_pipeline.named_steps['classifier']

# Transform the test data to get feature names and numerical values
X_test_transformed = prep.transform(X_test)

# Get feature names after one-hot encoding
cat_encoder = prep.named_transformers_['cat'].named_steps['onehot']
cat_features = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_features)

# Convert transformed data to DataFrame for SHAP
# Handle sparse matrices from OneHotEncoder
if hasattr(X_test_transformed, 'toarray'):
    X_test_transformed = X_test_transformed.toarray()
    
X_test_df = pd.DataFrame(X_test_transformed, columns=all_feature_names)

# Initialize SHAP explainer
# Use TreeExplainer for tree-based models, otherwise Linear Explainer
if best_model_name in ['Random Forest', 'XGBoost']:
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_test_df)
    
    # Random Forest returns a list of shap values for each class, we want class 1
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
else:
    explainer = shap.LinearExplainer(clf, X_test_df)
    shap_values = explainer.shap_values(X_test_df)

# Plot SHAP summary
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df, feature_names=all_feature_names, show=False)
plt.title(f"SHAP Summary Plot for {best_model_name}")
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=150)
plt.show()

## Business Interpretation & Recommendations

Based on the SHAP analysis of the best-performing model (which optimizes for AUC and handles class imbalance):

1. **Recency is King:** The most significant driver of churn is `recency`. As expected, customers who haven't purchased in a long time have a high positive SHAP value, strongly pushing the prediction towards "Churn" (1). **Action:** Implement automated re-engagement email campaigns at the 30-day and 60-day marks before they hit the 90-day churn threshold.
2. **Frequency and Monetary Value:** High frequency and high monetary values push the prediction toward "Retained" (0). Loyal, high-spending customers are far less likely to churn. **Action:** Reward high-frequency buyers with a VIP loyalty program to maintain their retention.
3. **Review Scores:** Lower average review scores increase churn probability. A bad experience directly translates to lost future revenue. **Action:** Create an automated support ticket for any review score < 3, ensuring proactive customer service intervention to salvage the relationship.
4. **Category / State Impact:** Certain states and product categories may show distinct churn behavior. For example, if a specific state is associated with higher churn, it could point to logistical issues (e.g., slow shipping times causing dissatisfaction). **Action:** Correlate geographic churn spikes with delivery performance metrics to identify operational bottlenecks.